In [ ]:
import numpy
import random
import pandas
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [ ]:
# Define random seeds for reproducibility
SEED: int = 42
random.seed(SEED)
numpy.random.seed(SEED)

In [ ]:
# Arrhythmia dataset
# https://archive.ics.uci.edu/dataset/5/arrhythmia

# There are 279 features and the 280th column is the target variable; generate column names.
column_names: list[str] = [f"feature_{i}" for i in range(1, 280)] + ["Class"]

# header=None + names=column_names tells pandas that the CSV has no header
# row and asks it to use the names we generated.
df: pandas.DataFrame = pandas.read_csv("arrhythmia.data", header=None, names=column_names)
print(df.head())

print(f"[*] Original dataset shape: {df.shape}")

# --- 1. HANDLE MISSING VALUES ---
print("[*] Replacing missing-value tokens ('?') with NaN...")
df.replace('?', numpy.nan, inplace=True)

# feature_14 (index 13 in the original description, the 'J' vector) has 376 missing values
# out of 452. The literature recommends dropping it; otherwise imputation is dominated by it.
print("[*] Dropping the J-vector column (feature_14) due to excessive missingness...")
df.drop(columns=['feature_14'], inplace=True)

# Convert every column to numeric (the '?' strings caused some columns to be object-typed).
df = df.apply(pandas.to_numeric)

# --- 2. BINARIZE THE TARGET VARIABLE ---
# Original: 1 = Normal, 2..16 = various arrhythmias
# Binarized target: 0 = Normal, 1 = Arrhythmia
print("[*] Binarizing the target (Outcome): 0 (Normal) vs 1 (Arrhythmia)...")
df['Outcome'] = (df['Class'] > 1).astype(int)
df.drop(columns=['Class'], inplace=True)  # drop the original multi-class column

feature_cols = [c for c in df.columns if c != 'Outcome']

# --- 3. STRATIFIED TRAIN / TEST SPLIT (before any *fitted* preprocessing) ---
# IMPORTANT: split FIRST, so every preprocessing step that learns a statistic
# from the data (here: the median used for imputation) is fit on the TRAINING
# partition only. Fitting the imputer on the full dataset -- as an earlier
# version of this notebook did -- leaks test-set feature distributions into the
# training data. Because the split is driven solely by the row index and the
# already-computed `Outcome` labels (not by any feature value), the resulting
# train/test partition is identical to one taken after imputation; only the
# imputed values change, and they are now leakage-free.
print("[*] Splitting into train and test sets (stratified 70/30 split)...")
train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    random_state=SEED,
    stratify=df["Outcome"]  # keeps the class ratio of 0 and 1 identical in both subsets
)
train_df = train_df.copy()
test_df = test_df.copy()

# --- 4. IMPUTE THE REMAINING MISSING VALUES (median, fit on TRAIN only) ---
# Learn the medians on the training partition, then apply those SAME medians to
# both partitions. This is the leakage-free equivalent of the previous
# full-dataset fit_transform.
print("[*] Imputing remaining missing values (median imputation, fit on train only)...")
imputer = SimpleImputer(strategy='median')
train_df[feature_cols] = imputer.fit_transform(train_df[feature_cols])
test_df[feature_cols] = imputer.transform(test_df[feature_cols])

# --- 5. BINARIZE NEAR-BINARY COLUMNS ---
# Deterministic per-row mapping (no fitted statistic), applied to each
# partition independently. feature_2 (Sex) is already encoded as 0/1; if there
# were multi-category string columns we would call pd.get_dummies() here.
binary_like = ['feature_56', 'feature_92', 'feature_104', 'feature_139',
               'feature_195', 'feature_225', 'feature_235', 'feature_264']
for col in binary_like:
    train_df[col] = (train_df[col] != 0).astype(int)
    test_df[col] = (test_df[col] != 0).astype(int)

print(f"[*] Train set shape: {train_df.shape} | Arrhythmia ratio: {train_df['Outcome'].mean():.2%}")
print(f"[*] Test set shape:  {test_df.shape} | Arrhythmia ratio: {test_df['Outcome'].mean():.2%}")

# --- 6. SAVE TO CSV ---
print("[*] Saving CSV files...")
train_csv_path = "arrhythmia_preprocessed_train_data.csv"
test_csv_path  = "arrhythmia_preprocessed_test_data.csv"

train_df.to_csv(train_csv_path, index=False, float_format="%.6f")
test_df.to_csv(test_csv_path, index=False, float_format="%.6f")

print("[*] Done -- CSV files saved.")